<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [1]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

**Weak — "add a search feature":** proposed a new `src/search/` package, a
`SearchIndex` class, a `rapidfuzz` dependency, and a CLI flag. It never opened
`tools.py`, so it did not notice `search_documents` already exists, and its test
asserted against its own in-memory fixture rather than the corpus.

**Project-aware — "read AGENTS.md, then propose a plan…":** came back with one
file (`src/bootcamp_agent/tools.py`), no new dependency, and a question: should
`tags` be AND or OR? It quoted the "keep tool inputs narrow and validate them at
the boundary" rule back at me and proposed the `ToolError` message shape that the
other two tools already use.

**The difference:** the weak prompt produced *architecture*; the project-aware one
produced a *diff plus a question*. The policy file is what turned invention into
a clarifying question.


## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Run `uv run pytest -q` and read failures before pasting them back.
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** it filtered by tag *after* `retrieve(...)` returned
the top-k hits, which silently turns `max_results=3` into "up to 3, often 0".
Then, asked to fix it, it offered to raise `MAX_SEARCH_RESULTS` from 5 so more
survived the filter. I rejected both: the first is a correctness bug disguised as
a one-line diff, the second widens a hard cap the caller is not allowed to exceed
in order to hide it. The fix was to filter the corpus *before* retrieval.


## 3. Improve the instructions

The assistant assumed `tags=["a","b"]` meant AND, and it assumed it was free to
change `MAX_SEARCH_RESULTS` because that constant lives in the one file it was
allowed to touch. Two sentences for `AGENTS.md`:

- *"A hard cap (`MAX_*`) is a contract, not a tunable. Never change one to make a
  new feature fit — say the feature does not fit."*
- *"When a new argument has more than one reasonable semantics (AND vs OR,
  inclusive vs exclusive), stop and ask before implementing; do not pick one and
  document it after."*

Instructions are code — see `docs/guides/harness-engineering.md`.


## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.

## 4. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [3]:
loop = {
    "plan_approved": "Plan limited to src/bootcamp_agent/tools.py plus one new test file: add an optional `tags` argument to search_documents, filter the corpus BEFORE calling retrieve so top_k still returns `capped` hits, leave MAX_SEARCH_RESULTS and the other two tools untouched.",
    "diff_inspected": "14 added / 2 removed lines in tools.py, read line by line: the widened signature (tags: Sequence[str] | None = None), a case-insensitive tag match building `pool`, a ToolError that names the known tags when the filter empties the pool, and retrieve(query, pool, top_k=capped) instead of retrieve(query, documents, ...). Plus tests/test_tags_filter.py with two cases. uv run pytest -q tests/ -> 2 passed; uv run ruff check -> All checks passed.",
    "rejected_change": "The first attempt filtered AFTER retrieval — it called retrieve(query, documents, top_k=capped) and then dropped the hits whose doc tags did not match. I rejected that and moved the filter before retrieve. I also rejected raising MAX_SEARCH_RESULTS from 5 to compensate for the thinner result set.",
    "why_rejected": "Filtering after retrieval silently breaks the max_results contract: ask for 3 and you can get 0 while matching documents sit just outside the top-k window, so the cap stops meaning what the tool description says it means. Raising the cap was out of scope — it changes a hard bound the caller cannot exceed, to paper over a bug in my own ordering, in a session whose budget is one file and no new behaviour.",
    "risks": "Remaining risks: (1) tag matching is case-insensitive set intersection, i.e. OR not AND — a caller asking for tags=['rag','safety'] gets either, which is not stated anywhere; (2) the empty-pool ToolError enumerates every tag in the corpus, so the message grows with the corpus and leaks the whole tag vocabulary to the model; (3) no test pins the OR semantics or the ordering fix, so a later refactor can reintroduce filter-after-retrieve and stay green; (4) the tool description string still does not mention `tags`, so a model reading the registry cannot discover the argument.",
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")


plan_approved      Plan limited to src/bootcamp_agent/tools.py plus one new t
diff_inspected     14 added / 2 removed lines in tools.py, read line by line:
rejected_change    The first attempt filtered AFTER retrieval — it called ret
why_rejected       Filtering after retrieval silently breaks the max_results 
risks              Remaining risks: (1) tag matching is case-insensitive set 


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [4]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [5]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True